# ECSE 415 Final Project

Path variables are defined in the next code cell:


External libraries used:
- OpenCV (`cv2`), NumPy, Matplotlib, Pandas
- PyTorch / torchvision
- scikit-learn
- scikit-image


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from skimage.feature import hog
from skimage import color
from skimage.transform import resize

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────

# Relative to the notebook, which should be in 415-Final-Project/
BASE_DIR  = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR  = os.path.join(BASE_DIR, "ecse-415-winter-2026-dog-vs-cat-classification", "train", "train")
CATS_DIR  = os.path.join(DATA_DIR, "cats")
DOGS_DIR  = os.path.join(DATA_DIR, "dogs")

# Quick sanity check — will tell you immediately if something is wrong
for path, name in [(DATA_DIR, "Data"), (CATS_DIR, "Cats"), (DOGS_DIR, "Dogs")]:
    if os.path.exists(path):
        print(f"✅ {name} folder found: {path}")
    else:
        print(f"❌ {name} folder NOT found: {path}")

# Count images
n_cats = len([f for f in os.listdir(CATS_DIR) if f.endswith('.jpg')])
n_dogs = len([f for f in os.listdir(DOGS_DIR) if f.endswith('.jpg')])
print(f"\n📁 Found {n_cats} cat images and {n_dogs} dog images")

In [ ]:
IMAGE_SIZE      = (128, 128)  # all images resized to this before HOG
ORIENTATIONS    = 9           # number of gradient direction bins (0°-180°)
PIXELS_PER_CELL = (8, 8)      # size of each HOG cell in pixels
CELLS_PER_BLOCK = (2, 2)      # how many cells per normalization block

def extract_hog_features(image_path):
    """
    Takes a path to an image file and returns its HOG feature vector.
    
    Steps:
    1. Open the image and force RGB (some images are grayscale or RGBA)
    2. Resize to fixed size so all HOG vectors are the same length
    3. Convert to grayscale (HOG works on intensity, not color)
    4. Compute HOG — returns a flat 1D numpy array
    """
    img = Image.open(image_path).convert('RGB')
    img = np.array(img.resize(IMAGE_SIZE))  # now shape (128, 128, 3)
    img_gray = color.rgb2gray(img)          # now shape (128, 128)
    

    features = hog(
        img_gray,
        orientations=ORIENTATIONS,          # 9 bins per cell histogram
        pixels_per_cell=PIXELS_PER_CELL,    # each cell covers 8x8 pixels
        cells_per_block=CELLS_PER_BLOCK,    # normalize over 2x2 cell blocks
        block_norm='L2-Hys'                 # normalization method (standard)
    )
    
    return features  # 1D array, same length for every image

def load_dataset(cats_dir, dogs_dir):
    """
    Loads all images from cats and dogs folders, extracts HOG features,
    and returns:
        X : numpy array of shape (n_images, n_features)
        y : numpy array of shape (n_images,) with 0=cat, 1=dog
    """
    
    features_list = []  # will collect one HOG vector per image
    labels_list   = []  # will collect one label per image
    skipped       = 0   # count images we failed to process
    
    #start with cats (and check if jpg files are present) label = 0
    cat_files = [f for f in os.listdir(cats_dir) if f.endswith('.jpg')]
    
    print(f"Processing {len(cat_files)} cat images...")
    for fname in tqdm(cat_files, desc="Cats"):
        path = os.path.join(cats_dir, fname)
        try:
            feat = extract_hog_features(path)
            features_list.append(feat)
            labels_list.append(0)           # 0 = cat
        except Exception as e:
            # Some images in the dataset are corrupted — skip them silently
            skipped += 1
    
    #load dogs (and check if jpg files are present) label = 1
    dog_files = [f for f in os.listdir(dogs_dir) if f.endswith('.jpg')]
    
    print(f"\nProcessing {len(dog_files)} dog images...")
    for fname in tqdm(dog_files, desc="Dogs"):
        path = os.path.join(dogs_dir, fname)
        try:
            feat = extract_hog_features(path)
            features_list.append(feat)
            labels_list.append(1)           # 1 = dog
        except Exception as e:
            skipped += 1
    
    # ── Convert to numpy arrays ────────────────────────────────────────────────
    # SVM requires numpy arrays, not Python lists.
    # np.array(features_list) stacks all vectors into a 2D matrix:
    #   rows    = images
    #   columns = HOG feature values
    X = np.array(features_list)
    y = np.array(labels_list)
    
    print(f"\n✅ Dataset loaded successfully")
    print(f"   X shape : {X.shape}  (n_images × n_hog_features)")
    print(f"   y shape : {y.shape}  (n_images,)")
    print(f"   Cats    : {np.sum(y == 0)}")
    print(f"   Dogs    : {np.sum(y == 1)}")
    if skipped > 0:
        print(f"   Skipped : {skipped} corrupted images")
    
    return X, y

X, y = load_dataset(CATS_DIR, DOGS_DIR)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, #(20% test, 80% train from instructions)
    random_state=42,
    stratify=y #ensures both splits have the same cat/dog ratio otherwise we get an unbalanced split between cat and dog images
)

print(f"Split complete")
print(f"   Training set : {X_train.shape[0]} images")
print(f"   Test set     : {X_test.shape[0]} images")
print(f"   Train cats   : {np.sum(y_train == 0)}  |  Train dogs: {np.sum(y_train == 1)}")
print(f"   Test cats    : {np.sum(y_test  == 0)}  |  Test dogs : {np.sum(y_test  == 1)}")

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True
    ))
])

print("\nTraining SVM...")


svm_pipeline.fit(X_train, y_train)

print("SVM training Finished!")


y_pred = svm_pipeline.predict(X_test)

accuracy = np.mean(y_pred == y_test) * 100
print(f"\n📊 Quick accuracy check on test set: {accuracy:.2f}%")
print(f"   Correct   : {np.sum(y_pred == y_test)} / {len(y_test)}")
print(f"   Incorrect : {np.sum(y_pred != y_test)} / {len(y_test)}")


In [ ]:
import joblib
joblib.dump(svm_pipeline, 'hog_svm_model.pkl')
print("✅ Model saved")

# To reload it later:
# svm_pipeline = joblib.load('hog_svm_model.pkl')

In [ ]:
# Confusion matrix (required for section 3.2)

def plot_confusion_matrix(y_true, y_pred):
    
    #   cm[0][0] = TN  (cat correctly predicted as cat)
    #   cm[0][1] = FP  (cat wrongly predicted as dog)
    #   cm[1][0] = FN  (dog wrongly predicted as cat)
    #   cm[1][1] = TP  (dog correctly predicted as dog)
    cm = confusion_matrix(y_true, y_pred)
    
    tn, fp, fn, tp = cm.ravel()  
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("HOG + SVM Evaluation", fontsize=14)
    
    sns.heatmap(
        cm,
        annot=True,             # write the numbers inside the cells
        fmt='d',                # format as integer
        cmap='Blues',           # color scale
        xticklabels=['Cat', 'Dog'],
        yticklabels=['Cat', 'Dog'],
        ax=axes[0]
    )
    axes[0].set_title("Confusion Matrix (counts)")
    axes[0].set_ylabel("True Label")
    axes[0].set_xlabel("Predicted Label")
    
    # Right plot: percentages
    cm_percent = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(
        cm_percent,
        annot=True,
        fmt='.1f',              # one decimal place
        cmap='Blues',
        xticklabels=['Cat', 'Dog'],
        yticklabels=['Cat', 'Dog'],
        ax=axes[1]
    )
    axes[1].set_title("Confusion Matrix (% per true class)")
    axes[1].set_ylabel("True Label")
    axes[1].set_xlabel("Predicted Label")
    
    plt.tight_layout()
    plt.savefig('confusion_matrix_hog_svm.png', dpi=150)
    plt.show()
    

    print("=" * 50)
    print("CLASSIFICATION REPORT")
    print("=" * 50)
    print(classification_report(y_true, y_pred, target_names=['Cat', 'Dog']))
    
    print("=" * 50)
    print("DETAILED BREAKDOWN")
    print("=" * 50)
    print(f"  True Negatives  (cat → cat) : {tn}  Correct")
    print(f"  False Positives (cat → dog) : {fp}  False, cats called dogs")
    print(f"  False Negatives (dog → cat) : {fn}  False, dogs called cats")
    print(f"  True Positives  (dog → dog) : {tp}  Correct")
    print(f"\n  Total correct   : {tn + tp} / {len(y_true)}")
    print(f"  Total incorrect : {fp + fn} / {len(y_true)}")
    print(f"  Accuracy        : {(tn + tp) / len(y_true) * 100:.2f}%")
    
    
    print(f"\n  Check for Bias:")
    if fp > fn * 1.2:
        print(f"Model leans toward predicting Dog (FP={fp} >> FN={fn})")
    elif fn > fp * 1.2:
        print(f"Model leans toward predicting Cat (FN={fn} >> FP={fp})")
    else:
        print(f"Model is balanced (FP={fp} ≈ FN={fn})")



plot_confusion_matrix(y_test, y_pred)

